In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -q tensorflow opencv-python scikit-learn matplotlib

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.optimizers import Adam

2026-04-16 11:15:56.441368: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776338156.633750      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776338156.688202      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776338157.138186      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776338157.138228      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776338157.138231      55 computation_placer.cc:177] computation placer alr

In [3]:
IMG_SIZE = 224

def load_data(data_dir):
    images = []
    labels = []
    
    for label in ["benign", "malignant"]:
        path = os.path.join(data_dir, label)
        class_num = 0 if label == "benign" else 1
        
        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            labels.append(class_num)
    
    return np.array(images), np.array(labels)

X, y = load_data("/kaggle/input/datasets/pankaj4321/breakhis-400x/train")
X = X / 255.0

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [5]:
def attention_block(x):
    gap = layers.GlobalAveragePooling2D()(x)
    dense = layers.Dense(x.shape[-1]//2, activation='relu')(gap)
    dense = layers.Dense(x.shape[-1], activation='sigmoid')(dense)
    scale = layers.Reshape((1,1,x.shape[-1]))(dense)
    return layers.multiply([x, scale])

In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, EfficientNetB0

def attention_block(x):
    gap = layers.GlobalAveragePooling2D()(x)
    dense = layers.Dense(x.shape[-1]//2, activation='relu')(gap)
    dense = layers.Dense(x.shape[-1], activation='sigmoid')(dense)
    scale = layers.Reshape((1,1,x.shape[-1]))(dense)
    return layers.multiply([x, scale])

def segmentation_head(x):
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)

    x = layers.UpSampling2D((2,2))(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)

    x = layers.UpSampling2D((2,2))(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)

    x = layers.UpSampling2D((2,2))(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)

    x = layers.UpSampling2D((2,2))(x)
    x = layers.Conv2D(16, 3, padding='same', activation='relu')(x)

    x = layers.UpSampling2D((2,2))(x)

    output = layers.Conv2D(1, 1, activation='sigmoid', name="seg_output")(x)
    return output


def build_model():
    input_layer = layers.Input(shape=(224,224,3))
    
    # Backbone 1
    resnet = ResNet50(weights='imagenet', include_top=False, input_tensor=input_layer)
    x1 = resnet.output   # ~7x7
    
    # Backbone 2
    effnet = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_layer)
    x2 = effnet.output   # ~7x7
    
    # Concatenate
    x = layers.Concatenate()([x1, x2])
    
    # Attention
    x = attention_block(x)
    
    # Transformer-lite (SAFE version)
    c = x.shape[-1]
    x_flat = layers.Reshape((-1, c))(x)
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x_flat, x_flat)
    x = layers.Reshape((7, 7, c))(attn)   # fixed spatial size
    
    # ✅ Correct Segmentation Head
    seg_output = segmentation_head(x)
    
    # Classification Head
    cls = layers.GlobalAveragePooling2D()(x)
    cls = layers.Dense(128, activation='relu')(cls)
    cls_output = layers.Dense(1, activation='sigmoid', name="cls_output")(cls)
    
    model = models.Model(inputs=input_layer, outputs=[seg_output, cls_output])
    
    return model


model = build_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 50,615,973 (193.08 MB)

 Trainable params: 50,520,830 (192.72 MB)

 Non-trainable params: 95,143 (371.66 KB)

In [10]:
# def build_model():
#     input_layer = layers.Input(shape=(224,224,3))
    
#     # Backbone 1
#     resnet = ResNet50(weights='imagenet', include_top=False, input_tensor=input_layer)
#     x1 = resnet.output
    
#     # Backbone 2
#     effnet = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_layer)
#     x2 = effnet.output
    
#     # Concatenate
#     x = layers.Concatenate()([x1, x2])
    
#     # Attention
#     x = attention_block(x)
    
#     # Transformer-lite (simplified)
#     shape = x.shape
#     x_flat = layers.Reshape((-1, shape[-1]))(x)
#     attn = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x_flat, x_flat)
#     x = layers.Reshape((shape[1], shape[2], shape[-1]))(attn)
    
#     # Segmentation Head (U-Net style)
#     seg = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
#     seg = layers.UpSampling2D((4,4))(seg)
#     seg_output = layers.Conv2D(1, 1, activation='sigmoid', name="seg_output")(seg)
    
#     # Classification Head
#     cls = layers.GlobalAveragePooling2D()(x)
#     cls = layers.Dense(128, activation='relu')(cls)
#     cls_output = layers.Dense(1, activation='sigmoid', name="cls_output")(cls)
    
#     model = models.Model(inputs=input_layer, outputs=[seg_output, cls_output])
    
#     return model

# model = build_model()
# model.summary()

In [11]:
model.compile(
    optimizer=Adam(1e-4),
    loss={
        "seg_output": "binary_crossentropy",
        "cls_output": "binary_crossentropy"
    },
    metrics={
        "seg_output": ["accuracy"],
        "cls_output": ["accuracy"]
    }
)

In [12]:
history = model.fit(
    X_train,
    {"seg_output": X_train[:,:,:,0:1], "cls_output": y_train},
    validation_data=(X_test, {"seg_output": X_test[:,:,:,0:1], "cls_output": y_test}),
    epochs=10,
    batch_size=8
)

Epoch 1/10


I0000 00:00:1776338473.331327     133 service.cc:152] XLA service 0x7da530003bb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776338473.331369     133 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776338485.450635     133 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-16 11:21:42.639649: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-16 11:21:42.824434: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-16 11:21:43.447888: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accur

118/119 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - cls_output_accuracy: 0.7399 - cls_output_loss: 0.4972 - loss: 1.1016 - seg_output_accuracy: 0.0045 - seg_output_loss: 0.6044

2026-04-16 11:23:08.225735: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-16 11:23:08.410364: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-16 11:23:08.957039: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-16 11:23:09.161630: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


119/119 ━━━━━━━━━━━━━━━━━━━━ 245s 889ms/step - cls_output_accuracy: 0.7411 - cls_output_loss: 0.4957 - loss: 1.0998 - seg_output_accuracy: 0.0045 - seg_output_loss: 0.6041 - val_cls_output_accuracy: 0.6793 - val_cls_output_loss: 7.7780 - val_loss: 8.4158 - val_seg_output_accuracy: 0.0031 - val_seg_output_loss: 0.6726
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 16s 131ms/step - cls_output_accuracy: 0.8939 - cls_output_loss: 0.2848 - loss: 0.8488 - seg_output_accuracy: 0.0048 - seg_output_loss: 0.5640 - val_cls_output_accuracy: 0.3207 - val_cls_output_loss: 4.6851 - val_loss: 5.2946 - val_seg_output_accuracy: 0.0031 - val_seg_output_loss: 0.6006
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 16s 133ms/step - cls_output_accuracy: 0.9496 - cls_output_loss: 0.1514 - loss: 0.7194 - seg_output_accuracy: 0.0042 - seg_output_loss: 0.5681 - val_cls_output_accuracy: 0.6793 - val_cls_output_loss: 1.2502 - val_loss: 1.8433 - val_seg_output_accuracy: 0.0031 - val_seg_output_loss: 0.5940
Epoch 4/10
119/119 ━━━━

In [ ]:
model.evaluate(X_test, {"seg_output": X_test[:,:,:,0:1], "cls_output": y_test})

In [ ]:
plt.plot(history.history['cls_output_accuracy'], label='train acc')
plt.plot(history.history['val_cls_output_accuracy'], label='val acc')
plt.legend()
plt.show()

In [ ]:
pred_seg, pred_cls = model.predict(X_test[:5])

for i in range(5):
    plt.imshow(X_test[i])
    plt.title(f"Pred: {pred_cls[i][0]:.2f}")
    plt.show()
    
    plt.imshow(pred_seg[i].squeeze(), cmap='gray')
    plt.title("Segmentation")
    plt.show()